In [2]:
using CSV, DataFrames, Dates, JSON, XLSX

In [2]:
function computation_new(df_flights, nbr_ac)
    # Agrégation par jour pour obtenir les totaux quotidiens
    daily_stats = combine(groupby(df_flights, :DAY),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )
    
    # Calcul des moyennes par avion et par jour
    total_flying_time = sum(daily_stats.FLYING_TIME)
    total_takeoffs = sum(daily_stats.TAKEOFF)
    nbr_days = length(unique(df_flights.DAY))
    fh_ac_day = round(Int, total_flying_time / nbr_ac / nbr_days)
    tk_ac_day = round(Int, total_takeoffs / nbr_ac / nbr_days)
    fh_tk = round(Int, total_flying_time / total_takeoffs)
    
    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end

computation_new (generic function with 1 method)

In [ ]:
fold = "INSTANCES/instances_literature_xlsx/A_MTN_5/"
nbr_ac = 8
for i in 21:30
    file = "354FL_8A_"*string(i)*".xlsx"
    df_flights = DataFrame(XLSX.readtable(fold*file, "Data"))
    df_param = DataFrame(XLSX.readtable(fold*file, "Parameters"))
    #df_mstation = DataFrame(XLSX.readtable(fold*file, "M_stations"))
    df_ac = DataFrame(XLSX.readtable(fold*file, "Aircrafts"))
    
    new_df_fl = filter(row -> row.DAY in [23, 24, 25], df_flights)
    nbr_fl = nrow(new_df_fl)

    result = computation_new(new_df_fl, nbr_ac)
    df_param.FH_DAY = [result.fh_ac_day]
    df_param.FH_TK = [result.fh_tk]
    df_param.TK_DAY = [result.tk_ac_day]

    airport = unique(vcat(new_df_fl.ORIGIN_AIRPORT, new_df_fl.DESTINATION_AIRPORT))
    nbr_airport = length(airport) 
    df_mstation = DataFrame(MTN_STATIONS = airport)
    for j in 1:7
        cap_mat = [rand() < 0.2 ? 0 : rand(1:3) for l in 1:nbr_airport]
        df_mstation[!, "T_" * string(j)] = cap_mat
    end
    filename = fold*string(nbr_fl)*"FL_"*string(nbr_ac)*"A_"*string(i)*".xlsx"
    XLSX.openxlsx(filename, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(new_df_fl); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstation); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_ac); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end 

Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé


In [50]:
fold = "INSTANCES/instances_lit_small_xlsx/A_MTN_3/"
#= ex_instance = "INSTANCES/instances_literature_xlsx/A_MTN_1/354FL_8A_1.xlsx"
df_mstn_ex = DataFrame(XLSX.readtable(ex_instance, "M_stations")) =#
nbr_ac = 8
for i in 11:20
    file = "152FL_8A_"*string(i)*".xlsx"
    df_flights = DataFrame(XLSX.readtable(fold*file, "Data"))
    df_param = DataFrame(XLSX.readtable(fold*file, "Parameters"))
    df_mstation = DataFrame(XLSX.readtable(fold*file, "M_stations"))
    df_ac = DataFrame(XLSX.readtable(fold*file, "Aircrafts"))

    df_param.F = [2400]
    df_param.T = [30]
    df_param.D = [3] 
    df_param.NBR_TP = [3]

    for row in eachrow(df_ac)
        if row.INIT_FLYING_TIME > 0.0
            row.INIT_FLYING_TIME = rand(1080:5:2220)
            row.INIT_TAKEOFF = ceil(Int, row.INIT_FLYING_TIME /96 )
            row.INIT_FLYING_DAY = ceil(Int, row.INIT_FLYING_TIME / 800)
            
            println("FT = ", row.INIT_FLYING_TIME, " | TK = ", row.INIT_TAKEOFF, " | DY = ", row.INIT_FLYING_DAY)
        end 
    end 
    
    XLSX.openxlsx(fold*file, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(df_flights); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstation); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_ac); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end 

FT = 1135 | TK = 12 | DY = 2
FT = 1945 | TK = 21 | DY = 3
FT = 2080 | TK = 22 | DY = 3
Fichier xlsx créé
FT = 2145 | TK = 23 | DY = 3
FT = 1200 | TK = 13 | DY = 2
FT = 1095 | TK = 12 | DY = 2
Fichier xlsx créé
FT = 1140 | TK = 12 | DY = 2
FT = 2115 | TK = 23 | DY = 3
FT = 1095 | TK = 12 | DY = 2
Fichier xlsx créé
FT = 2140 | TK = 23 | DY = 3
FT = 1745 | TK = 19 | DY = 3
FT = 1125 | TK = 12 | DY = 2
Fichier xlsx créé
FT = 1450 | TK = 16 | DY = 2
FT = 1560 | TK = 17 | DY = 2
FT = 1270 | TK = 14 | DY = 2
Fichier xlsx créé
FT = 1960 | TK = 21 | DY = 3
FT = 1880 | TK = 20 | DY = 3
FT = 2190 | TK = 23 | DY = 3
Fichier xlsx créé
FT = 1405 | TK = 15 | DY = 2
FT = 2160 | TK = 23 | DY = 3
FT = 1470 | TK = 16 | DY = 2
Fichier xlsx créé
FT = 1330 | TK = 14 | DY = 2
FT = 1320 | TK = 14 | DY = 2
FT = 1510 | TK = 16 | DY = 2
Fichier xlsx créé
FT = 1970 | TK = 21 | DY = 3
FT = 2200 | TK = 23 | DY = 3
FT = 1720 | TK = 18 | DY = 3
Fichier xlsx créé
FT = 1335 | TK = 14 | DY = 2
FT = 1510 | TK = 16 | DY =

In [7]:
fold = "INSTANCES/instances_literature_xlsx/A_MTN_5/"
ex_instance = "INSTANCES/instances_literature_xlsx/A_MTN_1/354FL_8A_1.xlsx"
df_mstn_ex = DataFrame(XLSX.readtable(ex_instance, "M_stations"))
for i in 21:30
    file = "354FL_8A_"*string(i)*".xlsx"
    df_flights = DataFrame(XLSX.readtable(fold*file, "Data"))
    df_param = DataFrame(XLSX.readtable(fold*file, "Parameters"))
    df_mstation = DataFrame(XLSX.readtable(fold*file, "M_stations"))
    df_ac = DataFrame(XLSX.readtable(fold*file, "Aircrafts"))
    
    df_mstation = df_mstn_ex
    XLSX.openxlsx(fold*file, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(df_flights); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstation); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_ac); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end 

Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
Fichier xlsx créé
